In [3]:
from ultralytics import YOLO
import cv2

# Load YOLOv8 Model
model = YOLO("yolov8n.pt")   

# Input & Output Video
video_path = "traffic.mp4"          
output_path = "smart_vehicle_counting_output.mp4"

cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# Counting Line
line_y = height // 2

# Vehicle Classes 
vehicle_classes = {
    2: "Car",
    3: "Motorcycle",
    5: "Bus",
    7: "Truck"
}

# Counters
counted_ids = set()

car_count = 0
motorcycle_count = 0
bus_count = 0
truck_count = 0

# Process Video
while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        verbose=False
    )

    # Draw counting line
    cv2.line(frame, (0, line_y), (width, line_y), (0, 255, 255), 3)

    if results[0].boxes.id is not None:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        classes = results[0].boxes.cls.cpu().numpy().astype(int)

        for box, track_id, cls in zip(boxes, ids, classes):

            if cls not in vehicle_classes:
                continue

            x1, y1, x2, y2 = map(int, box)

            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            label = vehicle_classes[cls]

            # Draw bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Display label and tracking ID
            cv2.putText(
                frame,
                f"{label} ID:{track_id}",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

            # Center point
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)

            # Count once
            if abs(cy - line_y) < 5 and track_id not in counted_ids:

                counted_ids.add(track_id)

                if cls == 2:
                    car_count += 1
                elif cls == 3:
                    motorcycle_count += 1
                elif cls == 5:
                    bus_count += 1
                elif cls == 7:
                    truck_count += 1

    total = car_count + motorcycle_count + bus_count + truck_count

    # Display Counts
    cv2.putText(frame, f"Cars: {car_count}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    cv2.putText(frame, f"Motorcycles: {motorcycle_count}", (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    cv2.putText(frame, f"Buses: {bus_count}", (20, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    cv2.putText(frame, f"Trucks: {truck_count}", (20, 145),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    cv2.putText(frame, f"Total: {total}", (20, 180),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

    # Save frame
    writer.write(frame)

    # Show frame
    cv2.imshow("Smart Vehicle Counting System", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
writer.release()
cv2.destroyAllWindows()

print("=" * 40)
print("Processing Completed Successfully!")
print(f"Cars: {car_count}")
print(f"Motorcycles: {motorcycle_count}")
print(f"Buses: {bus_count}")
print(f"Trucks: {truck_count}")
print(f"Total Vehicles: {total}")
print(f"Processed video saved as: {output_path}")

Processing Completed Successfully!
Cars: 24
Motorcycles: 14
Buses: 1
Trucks: 14
Total Vehicles: 53
Processed video saved as: smart_vehicle_counting_output.mp4
